In [1]:
import os
import torch
import pandas as pd

from tqdm import tqdm

from stock_bpt import StockBPT, LinearModel, NaiveModel
from dataloader_builder import build_dataloaders
from setup import StockBPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor
from model_training import model_setup, train_model_cuda

from model_training import train_model_cuda, evaluate_model, evaluate_best_model
from model_analysis import test_model, print_loss_analysis, process_losses, format_num, process_result, store_result

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## MODEL TRAINING ---------------------------

In [3]:
torch.manual_seed(1234)
print(path_data_preprocessor)
dls, train_norms = build_dataloaders(path_data_preprocessor)

preprocessed_data/data_1min_2021_2026_v1
Building DataLoaders...
Train dataset samples: 291,104
Train loader batches:  1,137
Batch size:            256


In [4]:
optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 10

eval_bs = 1000

stockBPT, stockBPT_params, opt1, sca1, sch1 = model_setup(StockBPT, StockBPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, opt2, sca2, sch2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

Input Norm: torch.Size([12])|torch.Size([12])
Target Norm: torch.Size([4])|torch.Size([4])
3261440
5376


NaiveModel()

In [6]:
model_train_losses, model_val_losses = train_model_cuda(stockBPT, device, opt1, sca1, sch1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, opt2, sca2, sch2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


Continuing from previous checkpoint...
[] []


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 5:

Learning Rate: 4.00e-04



|█▋        | 16.7% (10:42) Evaluating model on validation data... (360/361) [2498/14988]:                 

Epoch 5:
Training Loss:
   (MAE) 0.37602129578590393
   ($$$) 0.002856701612472534
   (NLL) 1.1169971227645874
Validation Loss:
   (MAE) 0.39059871435165405
   ($$$) 0.0029674693942070007
   (NLL) 1.1252460479736328

Best Validation: 0.9814085960388184
----------------------------------------------------------------------------------------------------

Learning Rate: 4.00e-04



...

|████████  | 80.0% (05:59) Evaluating model on validation data... (360/361) [19984/24980]: 

Epoch 8:
Training Loss:
   (MAE) 0.5287942886352539
   ($$$) 0.004016523249447346
   (NLL) 1.968808650970459
Validation Loss:
   (MAE) 0.5407133102416992
   ($$$) 0.004107106477022171
   (NLL) 2.0694377422332764

Best Validation: 1.5501667261123657
----------------------------------------------------------------------------------------------------

Learning Rate: 1.00e-04



|█████████ | 90.0% (06:43) Training LinearModel-B1H1rp_v2... [22488/24980]:                

Epoch 9:
Training Loss:
   (MAE) 0.5252484083175659
   ($$$) 0.003989554941654205
   (NLL) 1.9527392387390137
Validation Loss:
   (MAE) 0.5373427867889404
   ($$$) 0.004081465303897858
   (NLL) 2.0467963218688965

Best Validation: 1.5501667261123657
----------------------------------------------------------------------------------------------------

Learning Rate: 1.00e-04



Epoch 10:
Training Loss:
   (MAE) 0.5229086875915527
   ($$$) 0.003971688915044069
   (NLL) 1.9362406730651855
Validation Loss:
   (MAE) 0.5345859527587891
   ($$$) 0.0040604411624372005
   (NLL) 2.024327278137207

Best Validation: 1.5501667261123657
----------------------------------------------------------------------------------------------------

Finished


## Model Analysis -------------------------

In [5]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, opt2, sca2, sch2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
bpt_losses = evaluate_best_model(stockBPT, device, opt1, sca1, sch1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
bpt_test_losses = test_model(dls["test"], stockBPT, device, eval_bs, analysis_pbar)


|███▏      | 31.7% (00:31) Evaluating model on validation data... (360/361) [1361/4287]:                  

[] []


|██████▎   | 63.5% (01:00) Evaluating model on validation data... (360/361) [2722/4287]: 

[] []


|██████████| 100.0% (04:03) Evaluating model on testing data... (67/68) [4287/4287]:     

In [6]:
for key, features in [("NLL", StockBPT_cfg["target_features"]),
                      ("MAE", StockBPT_cfg["target_features"]),
                      ("RMAE", StockBPT_cfg["target_features"]),
                      ("STD", [f"{feature}_std" for feature in StockBPT_cfg["target_features"]]),
                      ("RSTD", [f"{feature}_std" for feature in StockBPT_cfg["target_features"]]),
                      ("Z^2", StockBPT_cfg["target_features"])]:
    print_loss_analysis(process_losses(bpt_losses + bpt_test_losses +
                                       linear_losses + linear_test_losses +
                                       naive_losses + naive_test_losses, key), 
                                       [stockBPT.cfg["name"], linearModel.cfg["name"], naiveModel.cfg["name"]],
                                       [format_num(stockBPT_params), format_num(linearModel_params), "0"], 
                                       features, key)


--------------------------------------------------------------------------------------------------------------

NLL

--------------------------------------------------------------------------------------------------------------

                    o        h        l        c        
StockGPT-B1H1rp_v1: 3.3M
    Training:       0.8991   0.9588   0.9722   1.0125     >  0.9606
    Validation:     0.9171   0.9841   0.9949   1.0296     >  0.9814
    Testing:        0.9084   0.9706   0.9754   1.0112     >  0.9664
    
LinearModel-B1H1rp_v1: 5.4K
    Training:       1.6462   1.3515   1.3487   1.5923     >  1.4847
    Validation:     1.7719   1.3911   1.3869   1.6508     >  1.5502
    Testing:        1.7898   1.3768   1.3810   1.6685     >  1.5540
    
NaiveModel-B1_v1: 0
    Training:       1.8825   1.8832   1.8815   1.8821     >  1.8823
    Validation:     1.9610   1.9657   1.9468   1.9445     >  1.9545
    Testing:        1.9104   1.9112   1.9023   1.9017     >  1.9064
    

------------

In [ ]:
return

import importlib
import setup
importlib.reload(setup)
from setup import PATH_RESULTS_RESIDUALS

store_result(PATH_RESULTS_RESIDUALS, process_result(stockBPT, bpt_losses, bpt_test_losses, max_epochs))
store_result(PATH_RESULTS_RESIDUALS, process_result(linearModel, linear_losses, linear_test_losses, max_epochs))
store_result(PATH_RESULTS_RESIDUALS, process_result(naiveModel, naive_losses, naive_test_losses, max_epochs))

print(pd.read_parquet(PATH_RESULTS_RESIDUALS))

In [ ]:
from data_scrapper import scrape_data, get_all_tickers
from data_filler import fill_data
from data_preprocessor import preprocess_data
from setup import API_KEY, TIMEFRAME
import pandas_market_calendars as mcal

In [ ]:
from setup import ID, API_KEY, TIMEFRAME
from data_scrapper import get_all_tickers, scrape_data
from data_preprocessor import preprocess_data
import pandas_market_calendars as mcal

#all_tickers = get_all_tickers("raw_data/all_tickers_trimmed_1_30", API_KEY)
#scrape_data(API_KEY, "raw_data/data_5min_2026_2026", 250, all_tickers,
#            mcal.get_calendar("NYSE").schedule("2026-01-01", "2026-7-1").index, TIMEFRAME)
#fill_data(f"raw_data/data_{ID}min_2026_2026", f"filled_raw_data/data_{ID}min_2026_2026",
#          mcal.get_calendar("NYSE").schedule("2026-01-01", "2026-8-1").index, False)

preprocess_data(f"filled_raw_data/data_{ID}min_2026", f"preprocessed_data/data_{ID}min_2026_2026_v2",
                mcal.get_calendar("NYSE").schedule("2026-01-01", "2026-8-1").index)

In [8]:
dls, train_norms = build_dataloaders("preprocessed_data/data_1min_2026_2026_v1", False)

Building DataLoaders...


In [9]:
test_losses = test_model(dls["test"], stockBPT, device, eval_bs)
print(test_losses)

({'NLL': tensor([4.1586, 3.0627, 3.0824, 2.5917], device='cuda:0'), 'MAE': tensor([0.6832, 0.6896, 0.6538, 0.6551], device='cuda:0'), 'RMAE': tensor([0.0052, 0.0053, 0.0050, 0.0050], device='cuda:0'), 'STD': tensor([0.7192, 0.6933, 0.7170, 0.7115], device='cuda:0'), 'RSTD': tensor([0.0054, 0.0053, 0.0054, 0.0054], device='cuda:0'), 'Z^2': tensor([7.2719, 5.2241, 5.1880, 4.1553], device='cuda:0')},)
